In [ ]:
# ==============================
# 0) Install nnUNetv2 from GitHub
# ==============================
!git clone https://github.com/MIC-DKFZ/nnUNet.git
%cd nnUNet
!pip install -e .
%cd /kaggle/working

Cloning into 'nnUNet'...
remote: Enumerating objects: 14008, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 14008 (delta 0), reused 0 (delta 0), pack-reused 14006 (from 2)
Receiving objects: 100% (14008/14008), 8.61 MiB | 25.10 MiB/s, done.
Resolving deltas: 100% (10702/10702), done.
/kaggle/working/nnUNet
Obtaining file:///kaggle/working/nnUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Prepari

In [ ]:

# ==============================
# 1) Copy pretrained BRATS19 weights into nnUNet_results
# ==============================
import os, shutil

pretrained_root = "/kaggle/input/dataset002-brats19/Dataset002_BRATS19/nnUNetTrainer__nnUNetPlans__3d_fullres"
print("Inner pretrained content:", os.listdir(pretrained_root))

results_root = "/kaggle/working/nnUNet_results"
target_dir = os.path.join(
    results_root,
    "Dataset002_BRATS19",
    "nnUNetTrainer__nnUNetPlans__3d_fullres",
)

# clean target if it exists
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)

os.makedirs(target_dir, exist_ok=True)

def copytree(src, dst):
    if not os.path.exists(dst):
        os.makedirs(dst)
    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)
        if os.path.isdir(s):
            copytree(s, d)
        else:
            shutil.copy2(s, d)

copytree(pretrained_root, target_dir)

print("✅ Fixed target dir:", target_dir)
print("Now contents:", os.listdir(target_dir))
print("Folds:", [f for f in os.listdir(target_dir) if f.startswith("fold_")])

Inner pretrained content: ['fold_0', 'plans.json', 'fold_4', 'fold_1', 'fold_3', 'dataset_fingerprint.json', 'fold_2', 'dataset.json']
✅ Fixed target dir: /kaggle/working/nnUNet_results/Dataset002_BRATS19/nnUNetTrainer__nnUNetPlans__3d_fullres
Now contents: ['fold_4', 'fold_1', 'fold_0', 'fold_3', 'dataset.json', 'dataset_fingerprint.json', 'fold_2', 'plans.json']
Folds: ['fold_4', 'fold_1', 'fold_0', 'fold_3', 'fold_2']


In [ ]:
# ==============================
# 2) Environment variables for nnUNetv2
# ==============================
os.environ["nnUNet_raw"] = "/kaggle/working/nnUNet_raw_data_base"
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_results:", os.environ["nnUNet_results"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])


nnUNet_raw: /kaggle/working/nnUNet_raw_data_base
nnUNet_results: /kaggle/working/nnUNet_results
nnUNet_preprocessed: /kaggle/working/nnUNet_preprocessed


In [ ]:
# ==============================
# 3) Prepare ISLES22 dataset
# ==============================
import shutil
import json

dataset_id = "Dataset003_ISLES22DWI"
input_dataset_dir = "/kaggle/input/isles22-dwi/Dataset003_ISLES22DWI"
nnunet_dataset_dir = os.path.join(os.environ["nnUNet_raw"], dataset_id)

# Copy the dataset into nnUNet_raw
if os.path.exists(nnunet_dataset_dir):
    shutil.rmtree(nnunet_dataset_dir)
shutil.copytree(input_dataset_dir, nnunet_dataset_dir)

print(f"✅ Copied ISLES22 dataset to {nnunet_dataset_dir}")

# Check folder structure
print("Folders in dataset:", os.listdir(nnunet_dataset_dir))
print("ImagesTr:", os.listdir(os.path.join(nnunet_dataset_dir, "imagesTr"))[:5])
print("LabelsTr:", os.listdir(os.path.join(nnunet_dataset_dir, "labelsTr"))[:5])

#compress the files because nnunet needs compressed files
import os
import gzip
import shutil

def compress_nii_folder(folder_path):
    """
    Compress all .nii files in a folder to .nii.gz
    and remove the original .nii files.
    """
    nii_files = [f for f in os.listdir(folder_path) if f.endswith(".nii")]
    if not nii_files:
        print(f"No .nii files found in {folder_path}.")
        return

    for f in nii_files:
        src = os.path.join(folder_path, f)
        dst = os.path.join(folder_path, f + ".gz")
        with open(src, "rb") as f_in, gzip.open(dst, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(src)  # remove original uncompressed file
    print(f"✅ Compressed {len(nii_files)} files in {folder_path} to .nii.gz")

# Paths to your ISLES22 dataset
raw_base = "/kaggle/working/nnUNet_raw_data_base/Dataset003_ISLES22DWI"
imagesTr_folder = os.path.join(raw_base, "imagesTr")
labelsTr_folder = os.path.join(raw_base, "labelsTr")

# Compress both folders
compress_nii_folder(imagesTr_folder)
compress_nii_folder(labelsTr_folder)

# Verify a few files
print("Sample imagesTr files:", os.listdir(imagesTr_folder)[:5])
print("Sample labelsTr files:", os.listdir(labelsTr_folder)[:5])


✅ Copied ISLES22 dataset to /kaggle/working/nnUNet_raw_data_base/Dataset003_ISLES22DWI
Folders in dataset: ['imagesTr', 'labelsTr', 'dataset.json']
ImagesTr: ['ISLES22_241_0000.nii', 'ISLES22_249_0000.nii', 'ISLES22_081_0000.nii', 'ISLES22_070_0000.nii', 'ISLES22_248_0000.nii']
LabelsTr: ['ISLES22_013.nii', 'ISLES22_004.nii', 'ISLES22_239.nii', 'ISLES22_211.nii', 'ISLES22_099.nii']
✅ Compressed 250 files in /kaggle/working/nnUNet_raw_data_base/Dataset003_ISLES22DWI/imagesTr to .nii.gz
✅ Compressed 250 files in /kaggle/working/nnUNet_raw_data_base/Dataset003_ISLES22DWI/labelsTr to .nii.gz
Sample imagesTr files: ['ISLES22_032_0000.nii.gz', 'ISLES22_040_0000.nii.gz', 'ISLES22_075_0000.nii.gz', 'ISLES22_192_0000.nii.gz', 'ISLES22_249_0000.nii.gz']
Sample labelsTr files: ['ISLES22_145.nii.gz', 'ISLES22_193.nii.gz', 'ISLES22_223.nii.gz', 'ISLES22_022.nii.gz', 'ISLES22_063.nii.gz']


In [ ]:
#update json file
import json, os, glob

dataset_dir = "/kaggle/working/nnUNet_raw_data_base/Dataset003_ISLES22DWI"
imagesTr = os.path.join(dataset_dir, "imagesTr")
labelsTr = os.path.join(dataset_dir, "labelsTr")

image_files = sorted(glob.glob(os.path.join(imagesTr, "*_0000.nii.gz")))
training_list = []

for img in image_files:
    base = os.path.basename(img).replace("_0000.nii.gz", "")
    lbl = os.path.join("labelsTr", f"{base}.nii.gz")
    training_list.append({
        "image": os.path.join("imagesTr", os.path.basename(img)),
        "label": lbl
    })

dataset_json = {
    "channel_names": {"0": "DWI"},
    "labels": {"background": 0, "lesion": 1},
    "numTraining": len(training_list),
    "file_ending": ".nii.gz",
    "training": training_list
}

with open(os.path.join(dataset_dir, "dataset.json"), "w") as f:
    json.dump(dataset_json, f, indent=4)

print("✅ dataset.json fixed. NumTraining:", len(training_list))


✅ dataset.json fixed. NumTraining: 250


In [ ]:
# ==============================
# 5) Check for GPU
# ==============================
import torch

if not torch.cuda.is_available():
    print("\n❌ No GPU detected (torch.cuda.is_available() is False).")
    print("nnUNetv2_predict requires a CUDA GPU and will crash with:")
    print("  RuntimeError: Cannot access accelerator device when none is available.")
    print("\nIn Kaggle:")
    print("  • Go to 'Session options' (right panel) -> 'Accelerator' -> set to 'GPU'.")
    print("  • Then rerun the notebook from the top.\n")
    raise KeyboardInterrupt("Aborted because no GPU detected")
else:
    print("\n✅ GPU detected:", torch.cuda.get_device_name(0))
    print("Proceeding to run nnUNetv2_predict...\n")


✅ GPU detected: Tesla T4
Proceeding to run nnUNetv2_predict...



In [ ]:
#preprocessing
import subprocess

dataset_id = "3"          # integer dataset id
network = "3d_fullres"

cmd_preprocess = [
    "nnUNetv2_plan_and_preprocess",
    "-d", dataset_id,           # dataset id (integer or string number)
    "-c", network,              # configuration
    "--verify_dataset_integrity"
]

print("Running preprocessing command:", " ".join(cmd_preprocess))

# Stream logs
process = subprocess.Popen(
    cmd_preprocess,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")  # live logs

process.wait()
if process.returncode != 0:
    print("❌ Preprocessing failed")
else:
    print("✅ Preprocessing completed successfully")


Running preprocessing command: nnUNetv2_plan_and_preprocess -d 3 -c 3d_fullres --verify_dataset_integrity
Fingerprint extraction...
Dataset003_ISLES22DWI
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

100%|██████████| 250/250 [00:19<00:00, 12.80it/s]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [66. 83. 68.], 3d_lowres: [66, 83, 68]
2D U-Net configuration:
{'dat

In [ ]:
#install wandb
!pip install wandb

In [ ]:
#login to wandb
!wandb login 55e586b0cfaff103f709c9d53c299b5794406d27

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [ ]:
!wandb status

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [ ]:
import subprocess
import wandb
import os
import glob

# -----------------------------
# Configurations
# -----------------------------
dataset_id = "3"
network = "3d_fullres"
trainer_class = "nnUNetTrainer_20epochs"
plans = "nnUNetPlans"
use_npz = True
folds = [0, 1, 2, 3, 4]

# Paths
output_base_dir = "/kaggle/working/nnUNet_results/Dataset003_ISLES22DWI"  # predictions/logs folder
full_results_dir = os.path.join(output_base_dir, f"{trainer_class}__{plans}__{network}")

# -----------------------------
# Initialize WandB (minimal)
# -----------------------------
wandb.init(
    project="nnUNet_training",
    entity="noorulainasghar-fast",
    config={
        "dataset_id": dataset_id,
        "network": network,
        "trainer_class": trainer_class,
        "plans": plans,
        "folds": folds
    }
)

# -----------------------------
# Training loop
# -----------------------------
for fold in folds:
    fold_dir = f"nnUNet_trained_models/{dataset_id}/{network}/{trainer_class}/fold_{fold}"
    checkpoint = os.path.join(fold_dir, "checkpoint_best.pth")

    cmd_train = [
        "nnUNetv2_train",
        dataset_id,
        network,
        str(fold),
        "-tr", trainer_class,
        "-p", plans
    ]
    if os.path.exists(checkpoint):
        cmd_train += ["-c", checkpoint]

    if use_npz:
        cmd_train.append("--npz")

    print(f"\n=== Training fold {fold} ===")
    print("Command:", " ".join(cmd_train))

    with subprocess.Popen(
        cmd_train,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=0
    ) as process:
        for line in iter(process.stdout.readline, ''):
            print(line, end='')  # live logs

        process.stdout.close()
        return_code = process.wait()
        if return_code != 0:
            print(f"❌ Training fold {fold} failed")
        else:
            print(f"✅ Training fold {fold} completed successfully")

    # -----------------------------
    # Upload fold-specific results folder to W&B
    # -----------------------------
    fold_results_dir = os.path.join(full_results_dir, f"fold_{fold}")
    if os.path.exists(fold_results_dir):
        artifact = wandb.Artifact(f"fold_{fold}_nnUNet_results", type="dataset")
        artifact.add_dir(fold_results_dir)
        wandb.log_artifact(artifact)
        print(f"✅ nnUNet_results for fold {fold} uploaded to W&B")

# -----------------------------
# Upload full results folder after all folds
# -----------------------------
if os.path.exists(full_results_dir):
    artifact = wandb.Artifact("full_nnUNet_results", type="dataset")
    artifact.add_dir(full_results_dir)
    wandb.log_artifact(artifact)
    print(f"✅ Full nnUNet_results uploaded to W&B")

wandb.finish()


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 


=== Training fold 0 ===
Command: nnUNetv2_train 3 3d_fullres 0 -tr nnUNetTrainer_20epochs -p nnUNetPlans --npz
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `

wandb: Adding directory to artifact (/kaggle/working/nnUNet_results/Dataset003_ISLES22DWI/nnUNetTrainer_20epochs__nnUNetPlans__3d_fullres/fold_0)... 

✅ Training fold 0 completed successfully


Done. 0.9s


...

Done. 1.0s
wandb: Adding directory to artifact (/kaggle/working/nnUNet_results/Dataset003_ISLES22DWI/nnUNetTrainer_20epochs__nnUNetPlans__3d_fullres)... 

✅ nnUNet_results for fold 4 uploaded to W&B


Done. 9.5s


✅ Full nnUNet_results uploaded to W&B


wandb: uploading artifact full_nnUNet_results
wandb: 
wandb:                                                                                
wandb: 🚀 View run volcanic-snowflake-9 at: https://wandb.ai/noorulainasghar-fast/nnUNet_training/runs/brfugkjp
wandb: ⭐️ View project at: https://wandb.ai/noorulainasghar-fast/nnUNet_training
wandb: Synced 5 W&B file(s), 0 media file(s), 1568 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20251205_063434-brfugkjp/logs


In [ ]:

# # ==============================
# # 5) Run nnUNetv2 prediction (only if GPU is available)
# # ==============================
# import subprocess

# input_dir = imagesTs
# output_dir = "/kaggle/working/nnunetv2_brats19_infer"
# os.makedirs(output_dir, exist_ok=True)

# cmd = [
#     "nnUNetv2_predict",
#     "-i", input_dir,
#     "-o", output_dir,
#     "-d", "2",                 # Dataset002_BRATS19
#     "-c", "3d_fullres",
#     "-tr", "nnUNetTrainer",
#     "-p", "nnUNetPlans"
# ]

# print("Running:", " ".join(cmd))
# res = subprocess.run(cmd, text=True, capture_output=True)
# print("Return code:", res.returncode)
# print("STDOUT (first 50 lines):\n", "\n".join(res.stdout.splitlines()[:50]))
# print("STDERR (first 50 lines):\n", "\n".join(res.stderr.splitlines()[:50]))

# print("Output dir contents:", os.listdir(output_dir))


In [ ]:
# import subprocess
# import os
# import glob
# import nibabel as nib
# import numpy as np
# from medpy.metric import binary
# import wandb

# # -----------------------------
# # Configurations
# # -----------------------------
# dataset_id = "3"
# network = "3d_fullres"
# trainer_class = "nnUNetTrainer"
# folds = [0, 1, 2, 3, 4]

# input_folder = "/path/to/images"         # your validation images
# labelsTr_dir = "/path/to/labelsTr"       # ground truth labels
# output_pred_dir = "/path/to/predictions" # folder to save predictions
# use_npz = True

# # -----------------------------
# # Initialize WandB (minimal)
# # -----------------------------
# wandb.init(
#     project="nnUNet_evaluation",
#     entity="noorulainasghar-fast",
#     config={
#         "dataset_id": dataset_id,
#         "network": network,
#         "trainer_class": trainer_class,
#         "folds": folds
#     }
# )

# # -----------------------------
# # Evaluation loop
# # -----------------------------
# for fold in folds:
#     cmd_eval = [
#         "nnUNetv2_predict",
#         "-i", input_folder,
#         "-o", output_pred_dir,
#         "-t", dataset_id,
#         "-m", network,
#         "-f", str(fold),
#         "-tr", trainer_class
#     ]
#     if use_npz:
#         cmd_eval.append("--save_npz")

#     print(f"\n=== Evaluating fold {fold} ===")
#     print("Command:", " ".join(cmd_eval))

#     # Run prediction
#     with subprocess.Popen(
#         cmd_eval,
#         stdout=subprocess.PIPE,
#         stderr=subprocess.STDOUT,
#         text=True,
#         bufsize=0
#     ) as process:
#         for line in iter(process.stdout.readline, ''):
#             print(line, end='')  # live logs
#         process.stdout.close()
#         return_code = process.wait()
#         if return_code != 0:
#             print(f"❌ Evaluation fold {fold} failed")
#         else:
#             print(f"✅ Evaluation fold {fold} completed successfully")

# # -----------------------------
# # Compute metrics over validation set
# # -----------------------------
# pred_files = glob.glob(os.path.join(output_pred_dir, "*.nii.gz"))

# dice_scores = []
# iou_scores = []
# hd95_scores = []

# for pred_file in pred_files:
#     case_id = os.path.basename(pred_file).split(".nii.gz")[0]
#     gt_file = os.path.join(labelsTr_dir, f"{case_id}.nii.gz")
#     if not os.path.exists(gt_file):
#         print(f"GT not found for {case_id}")
#         continue

#     pred = nib.load(pred_file).get_fdata() > 0.5
#     gt = nib.load(gt_file).get_fdata() > 0.5

#     dice = binary.dc(pred, gt)
#     iou = binary.jc(pred, gt)
#     hd95 = binary.hd95(pred, gt)

#     dice_scores.append(dice)
#     iou_scores.append(iou)
#     hd95_scores.append(hd95)

# # -----------------------------
# # Print and log metrics to WandB
# # -----------------------------
# mean_dice = np.mean(dice_scores)
# mean_iou = np.mean(iou_scores)
# mean_hd95 = np.mean(hd95_scores)

# print("\nValidation metrics (mean over cases):")
# print("Dice:", mean_dice)
# print("IoU:", mean_iou)
# print("HD95:", mean_hd95)

# wandb.log({
#     "mean_dice": mean_dice,
#     "mean_iou": mean_iou,
#     "mean_hd95": mean_hd95
# })

# wandb.finish()